# Harness Basics — fasteval-langgraph

Deep dive into the harness API: wrapping graphs, inspecting `ChatResult`, configuring hooks, and graph introspection.

**What you'll learn:**

1. **Wrapping graphs** — `MessagesState` and plain `TypedDict` graphs
2. **ChatResult** — response, state, trace, nodes_ran
3. **Auto-detection** — how the harness adapts to your graph type
4. **Custom hooks** — `input_fn`, `output_fn`, `state_filter`, factories
5. **Graph introspection** — `.nodes`, `.has_messages_state`, `.entry`
6. **Tool trajectory** — extracting and evaluating tool calls via the harness

Every cell is **runnable**. LLM-as-judge metrics require an OpenAI API key.

---

## Setup

In [ ]:
!pip install -q fasteval-langgraph

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded from Colab Secrets.")
except (ImportError, Exception):
    if os.environ.get("OPENAI_API_KEY"):
        print("API key found in environment.")
    else:
        print(
            "WARNING: OPENAI_API_KEY not set.\n"
            "LLM-as-judge metrics will fail. Deterministic checks still work."
        )

In [ ]:
import fasteval as fe
from fasteval_langgraph import harness, GraphHarness
from fasteval.models import ExpectedTool, ToolCall

print("Ready.")

---

## 1. Wrapping a Graph

The `harness()` function wraps any compiled `StateGraph`. It auto-detects whether the graph uses `MessagesState` or a plain `TypedDict` and configures sensible defaults.

> **Collab Note:** `harness()` is a thin wrapper around `GraphHarness()`. Both are equivalent — `harness()` is the preferred entry point for readability.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, HumanMessage


class EchoState(MessagesState):
    call_count: int


def echo(state: EchoState) -> dict:
    last_msg = state["messages"][-1].content
    return {
        "messages": [AIMessage(content=f"Echo: {last_msg}")],
        "call_count": state.get("call_count", 0) + 1,
    }


builder = StateGraph(EchoState)
builder.add_node("echo", echo)
builder.add_edge(START, "echo")
builder.add_edge("echo", END)
echo_compiled = builder.compile(checkpointer=MemorySaver())

# Wrap with harness
echo_graph = harness(echo_compiled)
result = await echo_graph.chat("Hello world")
print(f"Response:  {result.response}")
print(f"State:     {result.state}")
print(f"Nodes ran: {result.nodes_ran}")

### Each `.chat()` creates a new thread

Calls are isolated — no state leaks between tests.

In [ ]:
r1 = await echo_graph.chat("First")
r2 = await echo_graph.chat("Second")

print(f"r1 call_count: {r1.state['call_count']}")
print(f"r2 call_count: {r2.state['call_count']}")
assert r1.state["call_count"] == 1
assert r2.state["call_count"] == 1
print("Each .chat() starts a fresh thread.")

### Plain TypedDict Graphs

The harness adapts automatically for graphs without `MessagesState`.

> **Collab Note:** For plain graphs, the default `input_fn` maps the message to `{"input": msg}` and `output_fn` returns `str(state)`. You can override these with custom hooks.

In [ ]:
from typing import TypedDict


class CalcState(TypedDict):
    input: str
    result: str


def calc(state: CalcState) -> dict:
    return {"result": f"Calculated: {state['input']}"}


calc_builder = StateGraph(CalcState)
calc_builder.add_node("calc", calc)
calc_builder.add_edge(START, "calc")
calc_builder.add_edge("calc", END)
calc_compiled = calc_builder.compile(checkpointer=MemorySaver())

calc_graph = harness(calc_compiled)
result = await calc_graph.chat("5 + 3")

print(f"Response:   {result.response}")
print(f"State:      {result.state}")
print(f"Nodes ran:  {result.nodes_ran}")

---

## 2. Inspecting ChatResult

`ChatResult` is a Pydantic model with four fields:

| Field | Type | Description |
|-------|------|-------------|
| `response` | `str` | Last AI message content (via `output_fn`) |
| `state` | `dict` | Filtered state snapshot (via `state_filter`) |
| `trace` | `List[NodeStep]` | Per-node execution trace in order |
| `nodes_ran` | `List[str]` | Node names in execution order |

In [ ]:
result = await echo_graph.chat("Inspecting ChatResult")

print("=== ChatResult ===")
print(f"response:   {result.response!r}")
print(f"state:      {result.state}")
print(f"nodes_ran:  {result.nodes_ran}")
print(f"trace:")
for step in result.trace:
    print(f"  node={step.node}, updates={dict(step.updates)}")

### Trace with Multi-Node Graphs

Let's build a router graph to see a richer trace.

In [ ]:
from langgraph.types import Command


class RouterState(MessagesState):
    intent: str
    docs: list
    plan: str


def classifier_node(state):
    last_msg = state["messages"][-1].content
    intent = "TROUBLESHOOTING" if "troubleshoot" in last_msg.lower() else "FAQ"
    return Command(update={"intent": intent}, goto="rag" if intent == "FAQ" else "planner")


def rag_node(state):
    return Command(update={"docs": [f"doc for: {state['intent']}"]}, goto="responder")


def planner_node(state):
    return Command(update={"plan": f"plan for: {state['intent']}"}, goto="responder")


def responder_node(state):
    context = state.get("docs") or [state.get("plan", "")]
    return {"messages": [AIMessage(content=f"Answer: {context[0]}")]}


rb = StateGraph(RouterState)
rb.add_node("classifier", classifier_node)
rb.add_node("rag", rag_node)
rb.add_node("planner", planner_node)
rb.add_node("responder", responder_node)
rb.add_edge(START, "classifier")
rb.add_edge("responder", END)
router_compiled = rb.compile(checkpointer=MemorySaver())

router_graph = harness(router_compiled)
result = await router_graph.chat("What is OAuth?")

print(f"Nodes ran: {result.nodes_ran}")
print(f"State:     {result.state}")
print("\nFull trace:")
for step in result.trace:
    print(f"  {step.node}: {dict(step.updates)}")

---

## 3. Auto-Detection

The harness inspects the graph's channel definitions to detect `MessagesState` automatically.

| Behavior | MessagesState | Plain TypedDict |
|----------|:---:|:---:|
| `input_fn` | Wraps as `HumanMessage` | Sets `{"input": msg}` |
| `output_fn` | Last `AIMessage.content` | `str(state)` |
| `state_filter` | Excludes `messages` | Identity (all keys) |

> **Collab Note:** Auto-detection saves you from writing boilerplate. You only need custom hooks when your graph has non-standard conventions.

In [ ]:
print(f"Echo graph (MessagesState): has_messages_state = {echo_graph.has_messages_state}")
print(f"Calc graph (TypedDict):     has_messages_state = {calc_graph.has_messages_state}")

# MessagesState: state excludes messages
echo_result = await echo_graph.chat("test")
print(f"\nEcho state keys: {list(echo_result.state.keys())}")

# Plain: state includes all keys
calc_result = await calc_graph.chat("1+1")
print(f"Calc state keys: {list(calc_result.state.keys())}")

---

## 4. Custom Hooks

Override any auto-detected behavior by passing functions to `harness()`.

> **Collab Note:** Hooks are the escape hatch for non-standard graphs. Most graphs work fine with auto-detection.

In [ ]:
# Custom input_fn: add metadata to every message
custom_graph = harness(
    echo_compiled,
    input_fn=lambda msg: {
        "messages": [HumanMessage(content=msg, metadata={"source": "test"})],
    },
)
result = await custom_graph.chat("Hello with metadata")
print(f"Response: {result.response}")

In [ ]:
# Custom output_fn for a plain state graph
custom_calc = harness(
    calc_compiled,
    output_fn=lambda state: state.get("result", "No result"),
)
result = await custom_calc.chat("7 * 6")
print(f"Custom output_fn: {result.response}")

# Custom state_filter: only keep 'result'
filtered_calc = harness(
    calc_compiled,
    state_filter=lambda state: {"result": state.get("result")},
)
result = await filtered_calc.chat("7 * 6")
print(f"Filtered state: {result.state}")

# Config factory: inject custom configurable values
config_graph = harness(
    echo_compiled,
    config_factory=lambda tid: {
        "configurable": {"thread_id": tid, "model": "gpt-4o"},
    },
)
result = await config_graph.chat("With config factory")
print(f"Config factory result: {result.response}")

---

## 5. Graph Introspection

The harness exposes properties for inspecting the graph structure.

> **Collab Note:** These properties are useful for assertions in tests: verify that expected nodes exist, check graph type, or confirm the entry point.

In [ ]:
print("=== Router Graph ===")
print(f"Nodes:              {router_graph.nodes}")
print(f"Has MessagesState:  {router_graph.has_messages_state}")
print(f"Entry point:        {router_graph.entry}")

print("\n=== Echo Graph ===")
print(f"Nodes:              {echo_graph.nodes}")
print(f"Has MessagesState:  {echo_graph.has_messages_state}")

print("\n=== Calc Graph ===")
print(f"Nodes:              {calc_graph.nodes}")
print(f"Has MessagesState:  {calc_graph.has_messages_state}")

---

## 6. Tool Trajectory via Harness

When your agent uses LangChain tools, extract tool calls from `result.trace` and evaluate them with fasteval's tool metrics.

> **Collab Note:** Our sample graphs don't use LangChain tools, so we demonstrate with simulated tool calls. The extraction pattern is the same for real agents.

In [ ]:
# Simulated tool calls (in a real agent, extract from result.trace)
simulated_tool_calls = [
    ToolCall(name="get_weather", arguments={"city": "London"}),
]


@fe.tool_call_accuracy(threshold=0.9)
@fe.tool_args_match(threshold=0.85)
def test_weather_tool():
    return fe.score(
        actual_output="The weather in London is 65F and cloudy.",
        tool_calls=simulated_tool_calls,
        expected_tools=[ExpectedTool(name="get_weather", args={"city": "London"})],
        input="What's the weather in London?",
    )


result = test_weather_tool()
print(f"Passed: {result.passed}  |  Score: {result.aggregate_score:.2f}")
for m in result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

### Fuzzy Argument Matching

For search queries where the exact text may vary, use fuzzy matching.

In [ ]:
@fe.tool_args_match(threshold=0.8, fuzzy=True)
def test_fuzzy_search_args():
    """Fuzzy matching allows semantically similar arguments to pass."""
    return fe.score(
        actual_output="Recent climate change developments...",
        tool_calls=[ToolCall(name="search_web", arguments={"query": "recent climate change news 2024"})],
        expected_tools=[ExpectedTool(name="search_web", args={"query": "climate change news"})],
        input="Find recent news about climate change",
    )


result = test_fuzzy_search_args()
print(f"Passed: {result.passed}  |  Score: {result.aggregate_score:.2f}")
for m in result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

---

## Next Steps

| Notebook | What you'll learn |
|----------|------------------|
| [Multi-turn Sessions](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langgraph-testing/sessions.ipynb) | `.session()`, conversation metrics, interrupt/resume |
| [Node Testing & Mocking](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langgraph-testing/node-mocking.ipynb) | `.node().run()`, `mock()`, isolated testing patterns |